# Generating Multi-Hop Reasoning Datasets with Source2Synth

You can also check this cookbook in colab [here](https://colab.research.google.com/drive/1RT6Oo1acqB5mRBT17BM8ISoXT6AYGi_s?usp=sharing)

<div class="align-center">
  <a href="https://www.camel-ai.org/"><img src="https://i.postimg.cc/KzQ5rfBC/button.png" width="150"></a>
  <a href="https://discord.camel-ai.org"><img src="https://i.postimg.cc/L4wPdG9N/join-2.png" width="150"></a>
  
⭐ <i>Star us on [*Github*](https://github.com/camel-ai/camel), join our [*Discord*](https://discord.camel-ai.org) or follow our [*X*](https://x.com/camelaiorg)</i>
</div>

## Introduction: The Multi-Hop Reasoning Challenge

**Multi-hop reasoning** is the ability to answer questions that require connecting multiple pieces of information through a chain of logical steps. Consider this question:

> *"How did the invention of the printing press contribute to the Protestant Reformation?"*

Answering this requires connecting several facts:
1. The printing press enabled mass production of books
2. This made written materials affordable and accessible
3. Martin Luther's 95 Theses could be widely distributed
4. Mass distribution enabled rapid spread of Reformation ideas

**Why Multi-Hop QA Datasets Matter:**
- **Training reasoning models**: LLMs need examples of logical reasoning chains
- **Evaluating RAG systems**: Tests if retrieval + reasoning actually works
- **Educational assessment**: Creates comprehension questions that test deep understanding
- **Benchmarking**: Standard datasets like HotpotQA use multi-hop questions

**The Problem:** Creating multi-hop QA datasets manually is extremely time-consuming. Each question requires:
- Identifying related facts in a document
- Constructing a logical chain between them
- Writing a question that requires traversing this chain
- Documenting the reasoning steps

**The Solution:** CAMEL's `Source2Synth` automates this entire process. Give it any text, and it generates high-quality multi-hop QA pairs complete with reasoning chains.

### What You'll Learn

In this cookbook, you'll learn how to:
- Transform any text into multi-hop QA pairs using Source2Synth
- Understand and interpret the generated reasoning chains
- Control quality with complexity scoring and filtering
- Process batches of documents efficiently
- Export datasets for training or evaluation

## Setup & Installation

In [ ]:
# Install CAMEL with all dependencies
!pip install "camel-ai[data_tools]==0.2.79"

In [ ]:
# Configure your API key
import os
from getpass import getpass

# You can set your API key directly or use getpass for security
os.environ["OPENAI_API_KEY"] = getpass("Enter your OpenAI API key: ")

Alternatively, if running on Colab, you could save your API keys and tokens as **Colab Secrets**, and use them across notebooks.

To do so, **comment out** the above **manual** API key prompt code block(s), and **uncomment** the following codeblock.

⚠️ Don't forget granting access to the API key you would be using to the current notebook.

In [ ]:
# import os
# from google.colab import userdata

# os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")

In [ ]:
# Import Source2Synth components
from camel.datagen.source2synth import UserDataProcessor, ProcessorConfig

# Additional imports we'll use
import json
import logging

# Configure logging to see progress
logging.basicConfig(
    level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s'
)

print("Setup complete!")

## Quick Start: See It In Action

Let's immediately see what Source2Synth can do. We'll process a single paragraph and examine the generated multi-hop QA pairs.

Here's a paragraph about the Industrial Revolution that contains several connected facts:

In [ ]:
# A text with naturally connected facts - perfect for multi-hop QA
sample_text = """
The Industrial Revolution began in Britain in the late 18th century, fundamentally 
transforming society. The invention of the steam engine by James Watt in 1769 
provided a reliable power source for factories. These factories enabled mass 
production of goods, replacing traditional cottage industries. As factories grew, 
they required large workforces, causing massive migration from rural areas to cities. 
This urbanization led to the rapid growth of industrial cities like Manchester and 
Birmingham. The concentrated populations in cities created new public health 
challenges, eventually leading to reforms in sanitation and housing. These reforms 
laid the groundwork for modern urban planning and public health systems.
"""

print("Sample text loaded. Length:", len(sample_text), "characters")

In [ ]:
# Create a simple configuration
config = ProcessorConfig(
    seed=42,  # For reproducibility
    min_length=50,  # Minimum text length to process
    max_length=2000,  # Maximum text length
    complexity_threshold=0.5,  # Minimum complexity score (0.0-1.0)
    dataset_size=10,  # Target dataset size
    use_ai_model=True,  # Use AI for generation (vs rule-based)
)

# Create the processor
processor = UserDataProcessor(config)

# Process the text
print("Processing text with Source2Synth...")
results = processor.process_text(sample_text, source="industrial_revolution")

print(f"\nGenerated {len(results[0]['qa_pairs'])} QA pairs!")

In [ ]:
# Let's examine the generated QA pairs in detail
def display_qa_pair(qa, index):
    """Display a QA pair with full reasoning chain."""
    print(f"{'=' * 70}")
    print(f"QA PAIR {index}")
    print(f"{'=' * 70}")
    print(f"\n📝 QUESTION:\n{qa['question']}")

    print(
        f"\n🔗 REASONING CHAIN ({len(qa.get('reasoning_steps', []))} steps):"
    )
    for i, step in enumerate(qa.get('reasoning_steps', []), 1):
        step_text = step.get('step', step) if isinstance(step, dict) else step
        print(f"   {i}. {step_text}")

    print(f"\n✅ ANSWER:\n{qa['answer']}")

    print(f"\n📚 SUPPORTING FACTS:")
    for i, fact in enumerate(qa.get('supporting_facts', []), 1):
        print(f"   {i}. {fact}")

    print(f"\n📊 Type: {qa.get('type', 'N/A')}")
    print()


# Display all generated QA pairs
for i, qa in enumerate(results[0]['qa_pairs'], 1):
    display_qa_pair(qa, i)

# Show metadata
print(f"{'=' * 70}")
print(f"METADATA")
print(f"{'=' * 70}")
print(f"Source: {results[0]['metadata']['source']}")
print(f"Complexity Score: {results[0]['metadata']['complexity']:.2f}")

### 🎯 What Just Happened?

Source2Synth analyzed the text and:

1. **Identified fact chains**: Found sequences of related information (steam engine → factories → migration → urbanization → health reforms)

2. **Generated multi-hop questions**: Created questions that require connecting 2+ facts to answer

3. **Produced reasoning chains**: Documented the logical steps needed to reach the answer

4. **Extracted supporting facts**: Listed the specific facts from the text that support each answer

5. **Scored complexity**: Evaluated how "hard" each question is based on reasoning depth

**This is the power of Source2Synth** - what would take hours manually is done in seconds!

## Understanding the Output Structure

Let's dive deeper into what Source2Synth produces.

In [ ]:
# Examine the full output structure
print("Output Structure:")
print(json.dumps(results[0], indent=2, default=str)[:2000] + "...")

### Output Components

Each processed text returns a dictionary with:

| Field | Description |
|-------|-------------|
| `text` | The preprocessed source text |
| `qa_pairs` | List of generated question-answer pairs |
| `metadata` | Source info, timestamp, complexity score |

Each QA pair contains:

| Field | Description |
|-------|-------------|
| `question` | The multi-hop question |
| `reasoning_steps` | Ordered list of logical steps to reach the answer |
| `answer` | The complete answer |
| `supporting_facts` | Facts from the source text that support the answer |
| `type` | Generation type (e.g., "multi_hop_qa") |

In [ ]:
# Analyze the generated QA pairs
qa_pairs = results[0]['qa_pairs']

# Statistics
num_pairs = len(qa_pairs)
avg_reasoning_steps = (
    sum(len(qa.get('reasoning_steps', [])) for qa in qa_pairs) / num_pairs
    if num_pairs > 0
    else 0
)
avg_supporting_facts = (
    sum(len(qa.get('supporting_facts', [])) for qa in qa_pairs) / num_pairs
    if num_pairs > 0
    else 0
)
avg_question_length = (
    sum(len(qa['question'].split()) for qa in qa_pairs) / num_pairs
    if num_pairs > 0
    else 0
)
avg_answer_length = (
    sum(len(qa['answer'].split()) for qa in qa_pairs) / num_pairs
    if num_pairs > 0
    else 0
)

print("📊 QA Pair Statistics:")
print(f"   • Total QA pairs: {num_pairs}")
print(f"   • Avg reasoning steps: {avg_reasoning_steps:.1f}")
print(f"   • Avg supporting facts: {avg_supporting_facts:.1f}")
print(f"   • Avg question length: {avg_question_length:.1f} words")
print(f"   • Avg answer length: {avg_answer_length:.1f} words")
print(f"   • Complexity score: {results[0]['metadata']['complexity']:.2f}")

## Processing Real-World Content

Now let's process multiple documents to create a larger dataset. We'll use diverse topics to demonstrate Source2Synth's versatility.

In [ ]:
# A diverse corpus covering different domains
corpus = [
    {
        "text": """
        Photosynthesis is the process by which plants convert sunlight into chemical energy.
        Chlorophyll in plant leaves absorbs light energy, primarily from the blue and red 
        portions of the spectrum. This energy drives the conversion of carbon dioxide and 
        water into glucose and oxygen. The glucose produced serves as the primary energy 
        source for plant growth and metabolism. Plants store excess glucose as starch for 
        later use. This process also releases oxygen as a byproduct, which is essential 
        for animal respiration. The oxygen released by photosynthesis over billions of 
        years created Earth's oxygen-rich atmosphere, enabling the evolution of complex 
        animal life.
        """,
        "source": "biology_photosynthesis",
    },
    {
        "text": """
        The development of the internet began with ARPANET in the 1960s, a project funded 
        by the U.S. Department of Defense. ARPANET introduced packet switching, which 
        allowed data to be broken into small packets and routed independently. This 
        technology proved more resilient than circuit switching used in telephone networks.
        In 1989, Tim Berners-Lee invented the World Wide Web while working at CERN. The 
        Web introduced hypertext, allowing documents to link to each other. The release 
        of the Mosaic browser in 1993 made the Web accessible to ordinary users. This 
        accessibility sparked rapid commercial adoption, leading to the dot-com boom of 
        the late 1990s. Today, the internet connects billions of devices and has 
        fundamentally transformed commerce, communication, and entertainment.
        """,
        "source": "technology_internet",
    },
    {
        "text": """
        Climate change is primarily driven by the greenhouse effect, where certain gases 
        trap heat in Earth's atmosphere. Carbon dioxide, released mainly by burning fossil 
        fuels, is the most significant greenhouse gas. Since the Industrial Revolution, 
        atmospheric CO2 has increased by over 50%. This increase has caused global average 
        temperatures to rise by approximately 1.1 degrees Celsius. Rising temperatures are 
        melting polar ice caps and glaciers, contributing to sea level rise. Coastal 
        communities face increased flooding risks, forcing some populations to relocate. 
        Additionally, changing weather patterns are affecting agricultural productivity, 
        threatening food security in vulnerable regions.
        """,
        "source": "environment_climate",
    },
    {
        "text": """
        The human brain contains approximately 86 billion neurons, each connected to 
        thousands of others through synapses. Neurons communicate through electrical 
        impulses and chemical neurotransmitters. Learning occurs when repeated activation 
        strengthens synaptic connections, a process called long-term potentiation. Memory 
        formation involves the hippocampus, which consolidates short-term memories into 
        long-term storage during sleep. Different types of memory are stored in different 
        brain regions: procedural memories in the cerebellum, emotional memories in the 
        amygdala. Damage to specific brain regions can result in selective memory 
        impairments, demonstrating the distributed nature of memory systems.
        """,
        "source": "neuroscience_memory",
    },
]

print(f"Prepared corpus with {len(corpus)} documents:")
for doc in corpus:
    print(f"  • {doc['source']} ({len(doc['text'])} chars)")

In [ ]:
# Configure for batch processing
batch_config = ProcessorConfig(
    seed=42,
    min_length=100,
    max_length=2000,
    complexity_threshold=0.5,
    dataset_size=20,
    use_ai_model=True,
)

batch_processor = UserDataProcessor(batch_config)

# Extract texts and sources
texts = [doc["text"] for doc in corpus]
sources = [doc["source"] for doc in corpus]

# Process the batch
print("Processing batch of documents...")
batch_results = batch_processor.process_batch(texts, sources=sources)

print(f"\nProcessed {len(batch_results)} documents successfully!")

In [ ]:
# Analyze batch results
print("\n📊 BATCH PROCESSING RESULTS")
print("=" * 60)

total_qa_pairs = 0
all_complexities = []
all_reasoning_steps = []

for result in batch_results:
    source = result['metadata']['source']
    num_qa = len(result['qa_pairs'])
    complexity = result['metadata']['complexity']

    total_qa_pairs += num_qa
    all_complexities.append(complexity)

    for qa in result['qa_pairs']:
        all_reasoning_steps.append(len(qa.get('reasoning_steps', [])))

    print(f"\n📄 {source}")
    print(f"   QA pairs: {num_qa}")
    print(f"   Complexity: {complexity:.2f}")

print("\n" + "=" * 60)
print("OVERALL STATISTICS")
print("=" * 60)
print(f"Total documents: {len(batch_results)}")
print(f"Total QA pairs: {total_qa_pairs}")
print(
    f"Average complexity: {sum(all_complexities) / len(all_complexities):.2f}"
)
print(
    f"Average reasoning steps: {sum(all_reasoning_steps) / len(all_reasoning_steps):.1f}"
)

In [ ]:
# Display a sample QA from each domain
print("\n🎯 SAMPLE QA FROM EACH DOMAIN")
print("=" * 70)

for result in batch_results:
    if result['qa_pairs']:
        source = result['metadata']['source']
        qa = result['qa_pairs'][0]  # First QA pair

        print(f"\n📚 Source: {source}")
        print(f"\n   Q: {qa['question']}")
        print(
            f"\n   A: {qa['answer'][:200]}..."
            if len(qa['answer']) > 200
            else f"\n   A: {qa['answer']}"
        )
        print(f"\n   Reasoning steps: {len(qa.get('reasoning_steps', []))}")
        print("-" * 70)

## Quality Control with Complexity Scoring

Source2Synth includes a sophisticated complexity scoring system to help you control the quality of generated QA pairs.

### How Complexity is Calculated

The complexity score (0.0 to 1.0) is a weighted combination of four factors:

| Factor | Weight | Rationale |
|--------|--------|----------|
| Reasoning steps | 40% | More steps = harder reasoning chain |
| Supporting facts | 30% | More facts = more information to connect |
| Question length | 15% | Longer questions often encode more constraints |
| Answer length | 15% | Longer answers indicate more complex explanations |

**Formula:**
```
complexity = 0.4 * min(steps/3, 1.0) 
           + 0.3 * min(facts/3, 1.0)
           + 0.15 * min(q_words/20, 1.0)
           + 0.15 * min(a_words/50, 1.0)
```

In [ ]:
# Analyze complexity distribution
def analyze_complexity(qa_pair):
    """Break down complexity score components."""
    steps = len(qa_pair.get('reasoning_steps', []))
    facts = len(qa_pair.get('supporting_facts', []))
    q_words = len(qa_pair['question'].split())
    a_words = len(qa_pair['answer'].split())

    # Calculate component scores
    step_score = min(steps / 3, 1.0) * 0.4
    fact_score = min(facts / 3, 1.0) * 0.3
    q_score = min(q_words / 20, 1.0) * 0.15
    a_score = min(a_words / 50, 1.0) * 0.15

    total = step_score + fact_score + q_score + a_score

    return {
        'reasoning_steps': (steps, step_score),
        'supporting_facts': (facts, fact_score),
        'question_words': (q_words, q_score),
        'answer_words': (a_words, a_score),
        'total': total,
    }


# Analyze a sample QA pair
sample_qa = (
    batch_results[0]['qa_pairs'][0]
    if batch_results[0]['qa_pairs']
    else results[0]['qa_pairs'][0]
)
analysis = analyze_complexity(sample_qa)

print("🔍 COMPLEXITY BREAKDOWN")
print("=" * 50)
print(f"\nQuestion: {sample_qa['question'][:80]}...\n")

for component, value in analysis.items():
    if component != 'total':
        count, score = value
        print(f"{component:20} | Value: {count:3} | Score: {score:.3f}")

print("-" * 50)
print(f"{'TOTAL COMPLEXITY':20} |       | Score: {analysis['total']:.3f}")

In [ ]:
# Visualize complexity distribution across the dataset
import matplotlib.pyplot as plt

# Collect all QA complexities
qa_complexities = []
qa_steps = []

for result in batch_results:
    for qa in result['qa_pairs']:
        analysis = analyze_complexity(qa)
        qa_complexities.append(analysis['total'])
        qa_steps.append(len(qa.get('reasoning_steps', [])))

# Create visualization
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Complexity distribution
axes[0].hist(
    qa_complexities,
    bins=10,
    edgecolor='black',
    alpha=0.7,
    color='steelblue',
)
axes[0].axvline(x=0.5, color='red', linestyle='--', label='Threshold (0.5)')
axes[0].set_xlabel('Complexity Score')
axes[0].set_ylabel('Count')
axes[0].set_title('Complexity Score Distribution')
axes[0].legend()

# Reasoning steps distribution
axes[1].hist(
    qa_steps,
    bins=range(1, max(qa_steps) + 2),
    edgecolor='black',
    alpha=0.7,
    color='forestgreen',
)
axes[1].set_xlabel('Number of Reasoning Steps')
axes[1].set_ylabel('Count')
axes[1].set_title('Reasoning Steps Distribution')

plt.tight_layout()
plt.show()


In [ ]:
# Demonstrate filtering by complexity
def filter_by_complexity(results, min_complexity=0.0, max_complexity=1.0):
    """Filter QA pairs by complexity range."""
    filtered = []
    for result in results:
        for qa in result['qa_pairs']:
            analysis = analyze_complexity(qa)
            if min_complexity <= analysis['total'] <= max_complexity:
                filtered.append(
                    {
                        'question': qa['question'],
                        'answer': qa['answer'],
                        'complexity': analysis['total'],
                        'reasoning_steps': len(qa.get('reasoning_steps', [])),
                        'source': result['metadata']['source'],
                    }
                )
    return filtered


# Filter for different difficulty levels
easy = filter_by_complexity(batch_results, 0.0, 0.5)
medium = filter_by_complexity(batch_results, 0.5, 0.7)
hard = filter_by_complexity(batch_results, 0.7, 1.0)

print("🎯 FILTERING BY COMPLEXITY")
print("=" * 40)
print(f"Easy (0.0-0.5):   {len(easy)} questions")
print(f"Medium (0.5-0.7): {len(medium)} questions")
print(f"Hard (0.7-1.0):   {len(hard)} questions")

## Configuration Guide

Source2Synth's behavior can be customized through `ProcessorConfig`. Here's a comprehensive guide to the available parameters.

### Configuration Parameters

| Parameter | Type | Default | Description |
|-----------|------|---------|-------------|
| `seed` | int | random | Random seed for reproducibility |
| `min_length` | int | 50 | Minimum text length to process (characters) |
| `max_length` | int | 512 | Maximum text length to process (characters) |
| `complexity_threshold` | float | 0.5 | Minimum complexity score (0.0-1.0) |
| `dataset_size` | int | 1000 | Target size for final dataset |
| `use_ai_model` | bool | True | Use AI model vs rule-based generation |

In [ ]:
# Configuration presets for different use cases

# 1. Quick prototyping - fast, lower quality
prototype_config = ProcessorConfig(
    seed=42,
    min_length=50,
    max_length=500,
    complexity_threshold=0.3,  # Accept simpler questions
    dataset_size=100,
    use_ai_model=True,
)

# 2. Production quality - balanced
production_config = ProcessorConfig(
    seed=42,
    min_length=100,
    max_length=1500,
    complexity_threshold=0.5,  # Standard threshold
    dataset_size=1000,
    use_ai_model=True,
)

# 3. High-quality benchmark - strict filtering
benchmark_config = ProcessorConfig(
    seed=42,
    min_length=200,
    max_length=2000,
    complexity_threshold=0.7,  # Only complex questions
    dataset_size=500,
    use_ai_model=True,
)

print("📋 Configuration Presets Defined:")
print("   • prototype_config - Quick testing")
print("   • production_config - Balanced quality")
print("   • benchmark_config - High complexity")

### Recommended Settings by Use Case

| Use Case | complexity_threshold | min_length | Notes |
|----------|---------------------|------------|-------|
| Training data (general) | 0.4-0.5 | 50 | Balance quantity and quality |
| RAG evaluation | 0.6-0.7 | 150 | Harder questions test reasoning |
| Educational quizzes | 0.3-0.5 | 100 | Accessible difficulty |
| Benchmark creation | 0.7+ | 200 | Only challenging questions |

## Saving & Applications

Now let's export the generated dataset and discuss practical applications.

In [ ]:
# Export to JSON format (training-ready)
def export_to_json(results, output_path):
    """Export results to a clean JSON format."""
    export_data = []

    for result in results:
        for i, qa in enumerate(result['qa_pairs']):
            export_data.append(
                {
                    "id": f"{result['metadata']['source']}_{i}",
                    "question": qa['question'],
                    "answer": qa['answer'],
                    "reasoning_steps": [
                        step.get('step', step)
                        if isinstance(step, dict)
                        else step
                        for step in qa.get('reasoning_steps', [])
                    ],
                    "supporting_facts": qa.get('supporting_facts', []),
                    "source": result['metadata']['source'],
                    "complexity": analyze_complexity(qa)['total'],
                    "type": qa.get('type', 'multi_hop_qa'),
                }
            )

    with open(output_path, 'w', encoding='utf-8') as f:
        json.dump(export_data, f, indent=2, ensure_ascii=False)

    return export_data


# Export the dataset
exported = export_to_json(batch_results, 'multi_hop_qa_dataset.json')

print(f"✅ Exported {len(exported)} QA pairs to 'multi_hop_qa_dataset.json'")
print("\nSample exported entry:")
print(json.dumps(exported[0], indent=2))

### Practical Applications

Your generated multi-hop QA dataset can be used for:

#### 1. **Fine-tuning Reasoning Models**
```python
# Use the dataset to train models on chain-of-thought reasoning
# Format: question + reasoning_steps + answer
training_prompt = f"""Question: {qa['question']}

Let me think step by step:
{chr(10).join(qa['reasoning_steps'])}

Answer: {qa['answer']}"""
```

#### 2. **RAG System Evaluation**
- Multi-hop questions require retrieving AND connecting multiple documents
- A RAG system that just does similarity search will struggle
- Use the `supporting_facts` field to verify correct retrieval

#### 3. **Educational Assessment**
- Generate comprehension questions from textbooks
- Use complexity scores to create difficulty tiers
- Reasoning steps provide answer explanations

#### 4. **Benchmark Creation**
- Filter to high-complexity questions for challenging benchmarks
- Diverse sources ensure broad coverage
- Supporting facts enable automatic scoring

## Best Practices & Conclusion

### Best Practices

**1. Corpus Quality Matters**
- Use texts with naturally connected facts (cause-effect, sequential events)
- Avoid bullet-point lists (hard to extract reasoning chains)
- Longer, flowing paragraphs work best

**2. Start with Lower Thresholds**
- Begin with `complexity_threshold=0.4` to see what's generated
- Increase threshold after inspecting output quality
- Very high thresholds (>0.8) may yield few results

**3. Validate Sample Outputs**
- Always inspect a sample of generated QA pairs
- Check that reasoning steps logically connect
- Verify answers are factually grounded in source text

**4. Diversify Your Sources**
- Mix different topics and domains
- Prevents overfitting to specific patterns
- Creates more robust evaluation/training data

**5. Use Appropriate Configurations**
- Prototyping: low threshold, small dataset
- Production: balanced settings
- Benchmarks: high threshold, strict filtering

### Summary

In this cookbook, you learned how to:

✅ Use Source2Synth to transform text into multi-hop QA pairs

✅ Understand the output structure: questions, reasoning chains, supporting facts

✅ Process batches of documents efficiently

✅ Control quality with complexity scoring and filtering

✅ Configure Source2Synth for different use cases

### Next Steps

- **Explore CAMEL's other data generation tools**: CoT Data Generation, Self-Instruct
- **Combine with CAMEL agents**: Use generated data to train or evaluate agents
- **Scale up**: Process your own document corpus
- **Contribute**: Share your generated datasets with the community!

### Resources

- [CAMEL Documentation](https://docs.camel-ai.org/)
- [CAMEL GitHub Repository](https://github.com/camel-ai/camel)
- [Join CAMEL Discord](https://discord.camel-ai.org)

---

⭐ If you found this cookbook helpful, please star [CAMEL on GitHub](https://github.com/camel-ai/camel)!